系統工具

In [1]:
!pip install ipywidgets
!pip install pandas 
!pip install matplotlib
!pip install seaborn
!pip install numpy 
!pip install scikit-learn


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

# ==========================================================
# 1. 讀取資料與篩選已就業學生 (核心變數與資料)
# ==========================================================
df = pd.read_csv('global_placement.csv')  # 請確保 CSV 檔和程式碼在同一個資料夾
df_placed = df[df['placement_status'] == 'Placed'].copy()

# ==========================================================
# 2. 訓練隨機森林預測模型
# ==========================================================
features = ['cgpa', 'backlogs', 'college_tier', 'country', 'university_ranking_band', 
            'internship_count', 'aptitude_score', 'communication_score', 'specialization', 'industry', 'internship_quality_score']
X = pd.get_dummies(df_placed[features], drop_first=True)
y = df_placed['salary']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# ==========================================================
# 3. 定義簡易預測工具核心函式 (解決 predict_student_salary_tool 找不到的問題)
# ==========================================================
def predict_student_salary_tool(user_input_dict):
    # 建立與訓練資料相同結構的空白 DataFrame
    input_df = pd.DataFrame(0, index=[0], columns=X.columns)
    
    # 填入數值型欄位
    for col in ['cgpa', 'backlogs', 'internship_count', 'aptitude_score', 'communication_score', 'internship_quality_score']:
        if col in user_input_dict:
            input_df[col] = user_input_dict[col]
            
    # 填入類別型欄位
    for col in ['college_tier', 'country', 'university_ranking_band', 'specialization', 'industry']:
        if col in user_input_dict:
            dummy_col = f"{col}_{user_input_dict[col]}"
            if dummy_col in input_df.columns:
                input_df[dummy_col] = 1
                
    # 進行模型預測
    predicted_salary = model.predict(input_df)[0]
    
    # 計算該薪資在全體已就業學生中的 PR 值
    pr_value = (df_placed['salary'] < predicted_salary).mean() * 100
    return predicted_salary, pr_value

# ==========================================================
# 4. 定義介面互動元件 (Widgets)
# ==========================================================
style = {'description_width': 'initial'}

cgpa_slider = widgets.FloatSlider(value=7.8, min=4.0, max=10.0, step=0.1, description='學業成績 (CGPA):', style=style)
backlogs_input = widgets.BoundedIntText(value=0, min=0, max=10, step=1, description='學科未過科數:', style=style)
tier_dropdown = widgets.Dropdown(options=['Tier 1', 'Tier 2', 'Tier 3'], value='Tier 2', description='學校階層:', style=style)
country_dropdown = widgets.Dropdown(options=['USA', 'Germany', 'UK', 'Canada', 'India'], value='Germany', description='目標就業國家:', style=style)
ranking_dropdown = widgets.Dropdown(options=['Top 100', '100-300', '300+'], value='100-300', description='學校排名區間:', style=style)
intern_slider = widgets.IntSlider(value=1, min=0, max=5, step=1, description='目前實習次數:', style=style)
aptitude_slider = widgets.FloatSlider(value=75.0, min=30.0, max=100.0, step=1.0, description='性向測驗分數:', style=style)
comm_slider = widgets.FloatSlider(value=70.0, min=30.0, max=100.0, step=1.0, description='溝通能力分數:', style=style)
spec_dropdown = widgets.Dropdown(options=['Data Science', 'AI/ML', 'Cybersecurity', 'Core CS', 'Cloud'], value='Data Science', description='專業領域:', style=style)
industry_dropdown = widgets.Dropdown(options=['Tech', 'Consulting', 'Healthcare', 'Finance', 'Manufacturing', 'Other'], value='Tech', description='目標行業:', style=style)
quality_slider = widgets.FloatSlider(value=6.0, min=1.0, max=10.0, step=0.1, description='實習質量分數:', style=style)

btn_predict = widgets.Button(description='開始評估潛在薪資', button_style='danger', style={'font_weight': 'bold'}, layout=widgets.Layout(width='95%', height='40px'))
output_plots = widgets.Output()

# ==========================================================
# 5. 當按鈕點擊時要執行的核心繪圖與預測邏輯
# ==========================================================
def on_button_click(b):
    with output_plots:
        clear_output(wait=True) # 清除舊圖
        
        # 抓取介面數值
        user_background = {
            'cgpa': cgpa_slider.value, 'backlogs': backlogs_input.value, 'college_tier': tier_dropdown.value,
            'country': country_dropdown.value, 'university_ranking_band': ranking_dropdown.value,
            'internship_count': intern_slider.value, 'aptitude_score': aptitude_slider.value,
            'communication_score': comm_slider.value, 'specialization': spec_dropdown.value,
            'industry': industry_dropdown.value, 'internship_quality_score': quality_slider.value
        }
        
        # 預測薪資
        my_salary, my_pr = predict_student_salary_tool(user_background)
        
        # 繪製圖表
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 11))
        sns.set_theme(style="whitegrid")
        plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei', 'Arial']
        plt.rcParams['axes.unicode_minus'] = False
        
        # ----- [圖一：落點分佈] -----
        sns.kdeplot(df_placed['salary'], fill=True, color="skyblue", alpha=0.4, linewidth=2, ax=ax1, label='全體畢業生薪資分佈')
        ax1.axvline(x=my_salary, color='crimson', linestyle='--', linewidth=2.5)
        
        density_y = ax1.get_lines()[0].get_ydata()
        density_x = ax1.get_lines()[0].get_xdata()
        idx = (np.abs(density_x - my_salary)).argmin()
        ax1.plot(my_salary, density_y[idx], marker='*', color='gold', markersize=18, markeredgecolor='black', label='您的預測落點')
        
        mae_offset = 8000
        ax1.text(my_salary + 3000, density_y[idx] * 0.6, 
                 f"【預測結果】\n估計年薪: ${my_salary:,.0f} USD\n預估範圍: ${my_salary-mae_offset:,.0f} ~ ${my_salary+mae_offset:,.0f}\n求職市場超越度: {my_pr:.1f}% (PR {int(my_pr)})", 
                 fontsize=10, fontweight='bold', bbox=dict(facecolor='ivory', alpha=0.9, boxstyle="round,pad=0.5", edgecolor='crimson'))
        ax1.set_title('【工具一】個人背景在全體就業市場之薪資落點分佈圖', fontsize=14, fontweight='bold')
        ax1.set_xlabel('年薪範圍 (USD)', fontsize=11)
        ax1.set_ylabel('人數密度', fontsize=11)
        ax1.set_xlim(20000, 130000)
        ax1.legend(loc='upper left')
        
        # ----- [圖二：加薪模擬] -----
        scenarios = ['1. 維維持現狀', '2. 多增加 1 次實習經驗', '3. 將學業成績(CGPA)提升 1.0']
        sal_1 = my_salary
        
        bg_more_intern = user_background.copy()
        bg_more_intern['internship_count'] = min(user_background['internship_count'] + 1, 5)
        sal_2, _ = predict_student_salary_tool(bg_more_intern)
        
        bg_better_cgpa = user_background.copy()
        bg_better_cgpa['cgpa'] = min(user_background['cgpa'] + 1.0, 10.0)
        sal_3, _ = predict_student_salary_tool(bg_better_cgpa)
        
        salary_results = [sal_1, sal_2, sal_3]
        colors_list = ['#7f8c8d', '#2ecc71', '#3498db']
        
        sns.barplot(x=salary_results, y=scenarios, palette=colors_list, width=0.4, ax=ax2)
        for i, val in enumerate(salary_results):
            increase = val - sal_1
            label_text = f"${val:,.0f} (↗ 加薪 ${increase:,.0f})" if increase > 0 else f"${val:,.0f} (起點基準)"
            ax2.text(val + 1500, i, label_text, va='center', fontweight='bold', fontsize=11)
            
        ax2.set_title('【工具二】不同履歷優化策略之「加薪模擬」對比圖', fontsize=14, fontweight='bold')
        ax2.set_xlabel('預估潛在年薪 (USD)', fontsize=11)
        ax2.set_xlim(20000, max(salary_results) * 1.3)
        
        plt.tight_layout()
        plt.show()

# 綁定按鈕點擊事件
btn_predict.on_click(on_button_click)

# ==========================================================
# 6. 使用網格排版並渲染 UI 介面
# ==========================================================
col1 = widgets.VBox([cgpa_slider, backlogs_input, tier_dropdown, country_dropdown, ranking_dropdown], layout=widgets.Layout(width='50%'))
col2 = widgets.VBox([intern_slider, quality_slider, aptitude_slider, comm_slider, spec_dropdown, industry_dropdown], layout=widgets.Layout(width='50%'))

ui_form = widgets.VBox([
    widgets.HTML(value="<h2> 跨國就業薪資智能評估工具 (預估模擬器)</h2><hr>"),
    widgets.HBox([col1, col2]),
    widgets.HTML(value="<br>"),
    btn_predict,
    widgets.HTML(value="<br>"),
    output_plots
], layout=widgets.Layout(padding='15px', border='1px solid #cbd5e1', border_radius='10px', background_color='#f8fafc'))

# 顯示整個表單
display(ui_form)